# Collect Survey Papers from Markdown arXiv URLs

This notebook reads `nlp_survey_papers.md`, extracts survey entries that contain arXiv URLs, fetches metadata from the arXiv API by ID, and saves merged outputs.

Outputs:
- `survey_papers_arxiv_links_extracted.csv`
- `survey_papers_arxiv_metadata.csv`
- `survey_papers_arxiv_metadata.json`
- `survey_papers_arxiv_missing_ids.csv`

In [1]:
# %pip install requests pandas
import time
from pathlib import Path
from urllib.parse import urlparse
import xml.etree.ElementTree as ET

import pandas as pd
import requests

BASE_URL = "http://export.arxiv.org/api/query"
MAX_RESULTS_PER_CALL = 50
REQUEST_DELAY_SECONDS = 3
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 3

project_root = None
for candidate in [Path("."), Path("..")]:
    if (candidate / "nlp_survey_papers.md").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise FileNotFoundError("Could not locate nlp_survey_papers.md from current working directory.")

SURVEY_MD_PATH = project_root / "nlp_survey_papers.md"
OUTPUT_DIR = project_root / "Dataset collection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTED_LINKS_CSV = OUTPUT_DIR / "survey_papers_arxiv_links_extracted.csv"
OUTPUT_CSV = OUTPUT_DIR / "survey_papers_arxiv_metadata.csv"
OUTPUT_JSON = OUTPUT_DIR / "survey_papers_arxiv_metadata.json"
MISSING_IDS_CSV = OUTPUT_DIR / "survey_papers_arxiv_missing_ids.csv"

ATOM_NS = "{http://www.w3.org/2005/Atom}"
ARXIV_NS = "{http://arxiv.org/schemas/atom}"

session = requests.Session()
session.headers.update({
    "User-Agent": "ResearchPaperCategorizer/1.0 (mailto:replace-with-your-email@example.com)"
})

print(f"Project root: {project_root}")
print(f"Input markdown: {SURVEY_MD_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Project root: /home/aman/storage/Python/Projects/Research Paper Categorizer
Input markdown: /home/aman/storage/Python/Projects/Research Paper Categorizer/nlp_survey_papers.md
Output dir: /home/aman/storage/Python/Projects/Research Paper Categorizer/Dataset collection


In [3]:
def clean_text(value: str) -> str:
    return " ".join((value or "").split())


def normalize_arxiv_id(raw_id: str) -> str:
    value = clean_text(raw_id).replace("arXiv:", "").strip().strip("/")
    if value.endswith(".pdf"):
        value = value[:-4]

    if "v" in value:
        base, maybe_version = value.rsplit("v", 1)
        if maybe_version.isdigit():
            value = base

    return value


def extract_arxiv_id_from_url(url: str) -> str:
    parsed = urlparse(url)
    if "arxiv.org" not in parsed.netloc.lower():
        return ""

    path = parsed.path.strip("/")
    if path.startswith("abs/"):
        raw_id = path[len("abs/"): ]
    elif path.startswith("pdf/"):
        raw_id = path[len("pdf/"): ]
    else:
        return ""

    return normalize_arxiv_id(raw_id)


def parse_markdown_papers(markdown_text: str) -> pd.DataFrame:
    rows = []
    current_list_name = ""
    current_topic = ""

    for line_no, raw_line in enumerate(markdown_text.splitlines(), start=1):
        line = raw_line.strip()

        if line.startswith("## The ") and "Paper List" in line:
            current_list_name = clean_text(line.replace("##", "").replace("Paper List", ""))
            continue

        if line.startswith("#### [") and "](#content)" in line:
            current_topic = clean_text(line[len("#### ["):].split("](#content)", 1)[0])
            continue

        if ". **" not in line or "[paper](" not in line:
            continue

        prefix = line.split(". ", 1)[0]
        if not prefix.isdigit():
            continue

        title_parts = line.split("**")
        if len(title_parts) < 3:
            continue

        md_title = clean_text(title_parts[1])
        after_title = clean_text(title_parts[2])
        source_note = clean_text(after_title.split("[paper](", 1)[0])
        paper_url = clean_text(line.split("[paper](", 1)[1].split(")", 1)[0])
        arxiv_id = extract_arxiv_id_from_url(paper_url)

        rows.append({
            "line_no": line_no,
            "list_name": current_list_name,
            "topic": current_topic,
            "md_title": md_title,
            "source_note": source_note,
            "paper_url": paper_url,
            "arxiv_id": arxiv_id,
            "is_arxiv_url": bool(arxiv_id),
        })

    return pd.DataFrame(rows)


def parse_arxiv_entry(entry) -> dict:
    arxiv_url = clean_text(entry.findtext(f"{ATOM_NS}id", default=""))
    versioned_id = arxiv_url.rsplit("/", 1)[-1] if arxiv_url else ""
    arxiv_id = normalize_arxiv_id(versioned_id)

    authors = []
    for author_node in entry.findall(f"{ATOM_NS}author"):
        name = clean_text(author_node.findtext(f"{ATOM_NS}name", default=""))
        if name:
            authors.append(name)

    categories = []
    for category_node in entry.findall(f"{ATOM_NS}category"):
        term = clean_text(category_node.attrib.get("term", ""))
        if term:
            categories.append(term)

    primary_category = ""
    primary_node = entry.find(f"{ARXIV_NS}primary_category")
    if primary_node is not None:
        primary_category = clean_text(primary_node.attrib.get("term", ""))
    if not primary_category and categories:
        primary_category = categories[0]

    pdf_url = ""
    for link_node in entry.findall(f"{ATOM_NS}link"):
        title = clean_text(link_node.attrib.get("title", "")).lower()
        href = clean_text(link_node.attrib.get("href", ""))
        if title == "pdf" and href:
            pdf_url = href
            break

    return {
        "arxiv_id": arxiv_id,
        "arxiv_id_versioned": versioned_id,
        "arxiv_title": clean_text(entry.findtext(f"{ATOM_NS}title", default="")),
        "arxiv_abstract": clean_text(entry.findtext(f"{ATOM_NS}summary", default="")),
        "published": clean_text(entry.findtext(f"{ATOM_NS}published", default="")),
        "updated": clean_text(entry.findtext(f"{ATOM_NS}updated", default="")),
        "authors": " | ".join(authors),
        "primary_category": primary_category,
        "categories": " ".join(sorted(set(categories))),
        "comment": clean_text(entry.findtext(f"{ARXIV_NS}comment", default="")),
        "doi": clean_text(entry.findtext(f"{ARXIV_NS}doi", default="")),
        "journal_ref": clean_text(entry.findtext(f"{ARXIV_NS}journal_ref", default="")),
        "arxiv_url": arxiv_url,
        "pdf_url": pdf_url,
    }


def fetch_arxiv_batch(arxiv_ids: list[str]) -> list[dict]:
    params = {"id_list": ",".join(arxiv_ids), "max_results": len(arxiv_ids)}

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            entries = root.findall(f"{ATOM_NS}entry")
            return [parse_arxiv_entry(entry) for entry in entries]
        except (requests.RequestException, ET.ParseError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                wait_seconds = attempt * 2
                print(f"Retrying after error: {exc} (waiting {wait_seconds}s)")
                time.sleep(wait_seconds)

    raise RuntimeError(f"arXiv request failed after {MAX_RETRIES} attempts: {last_error}")

In [4]:
markdown_text = SURVEY_MD_PATH.read_text(encoding="utf-8", errors="replace")
all_rows = parse_markdown_papers(markdown_text)
if all_rows.empty:
    raise RuntimeError("No numbered paper entries parsed from nlp_survey_papers.md")

link_rows = all_rows[all_rows["is_arxiv_url"]].copy()
if link_rows.empty:
    raise RuntimeError("No arXiv URLs found in paper entries.")

id_counts = (
    link_rows["arxiv_id"]
    .value_counts()
    .rename_axis("arxiv_id")
    .reset_index(name="occurrences_in_markdown")
)

link_rows = link_rows.merge(id_counts, on="arxiv_id", how="left")
link_rows.to_csv(EXTRACTED_LINKS_CSV, index=False)

unique_ids = sorted(id_counts["arxiv_id"].tolist())
print(f"Parsed entries: {len(all_rows):,}")
print(f"Rows with arXiv URLs: {len(link_rows):,}")
print(f"Unique arXiv IDs: {len(unique_ids):,}")

records = []
for start in range(0, len(unique_ids), MAX_RESULTS_PER_CALL):
    batch_ids = unique_ids[start : start + MAX_RESULTS_PER_CALL]
    batch = fetch_arxiv_batch(batch_ids)
    records.extend(batch)

    batch_number = start // MAX_RESULTS_PER_CALL + 1
    print(f"Batch {batch_number}: requested={len(batch_ids)} returned={len(batch)} total={len(records)}")

    if start + MAX_RESULTS_PER_CALL < len(unique_ids):
        time.sleep(REQUEST_DELAY_SECONDS)

metadata_df = pd.DataFrame(records)
if metadata_df.empty:
    raise RuntimeError("arXiv API returned zero records for extracted IDs.")

metadata_df = metadata_df.drop_duplicates(subset=["arxiv_id"]).copy()
merged_df = link_rows.merge(metadata_df, on="arxiv_id", how="left")
merged_df["found_in_arxiv_api"] = merged_df["arxiv_title"].notna()

ordered_cols = [
    "list_name",
    "topic",
    "line_no",
    "md_title",
    "source_note",
    "paper_url",
    "arxiv_id",
    "occurrences_in_markdown",
    "found_in_arxiv_api",
    "arxiv_id_versioned",
    "arxiv_title",
    "arxiv_abstract",
    "authors",
    "published",
    "updated",
    "primary_category",
    "categories",
    "comment",
    "doi",
    "journal_ref",
    "arxiv_url",
    "pdf_url",
]
merged_df = merged_df[ordered_cols].copy()

missing_ids = sorted(set(unique_ids) - set(metadata_df["arxiv_id"].dropna().tolist()))
missing_df = pd.DataFrame({"arxiv_id": missing_ids})

merged_df.to_csv(OUTPUT_CSV, index=False)
merged_df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)
missing_df.to_csv(MISSING_IDS_CSV, index=False)

print()
print(f"Saved extracted links: {EXTRACTED_LINKS_CSV}")
print(f"Saved merged CSV: {OUTPUT_CSV} ({len(merged_df):,} rows)")
print(f"Saved merged JSON: {OUTPUT_JSON}")
print(f"Saved missing ID list: {MISSING_IDS_CSV} ({len(missing_df):,} IDs)")

Parsed entries: 1,063
Rows with arXiv URLs: 901
Unique arXiv IDs: 859
Batch 1: requested=50 returned=50 total=50
Batch 2: requested=50 returned=50 total=100
Batch 3: requested=50 returned=50 total=150
Batch 4: requested=50 returned=50 total=200
Batch 5: requested=50 returned=50 total=250
Batch 6: requested=50 returned=50 total=300
Batch 7: requested=50 returned=50 total=350
Batch 8: requested=50 returned=50 total=400
Batch 9: requested=50 returned=50 total=450
Batch 10: requested=50 returned=50 total=500
Batch 11: requested=50 returned=50 total=550
Batch 12: requested=50 returned=50 total=600
Batch 13: requested=50 returned=50 total=650
Batch 14: requested=50 returned=50 total=700
Batch 15: requested=50 returned=50 total=750
Batch 16: requested=50 returned=50 total=800
Batch 17: requested=50 returned=50 total=850
Batch 18: requested=9 returned=9 total=859

Saved extracted links: /home/aman/storage/Python/Projects/Research Paper Categorizer/Dataset collection/survey_papers_arxiv_links_e

In [5]:
print("API match summary:")
print(merged_df["found_in_arxiv_api"].value_counts(dropna=False))

print()
print("Top primary categories:")
print(merged_df["primary_category"].fillna("<missing>").value_counts().head(15))

print()
missing_examples = merged_df[~merged_df["found_in_arxiv_api"]][["md_title", "paper_url", "arxiv_id"]].head(10)
print("Examples missing from API response:")
missing_examples

API match summary:
found_in_arxiv_api
True    901
Name: count, dtype: int64

Top primary categories:
primary_category
cs.LG      307
cs.CL      298
cs.CV      121
cs.IR       40
cs.AI       32
stat.ML     18
cs.CR       14
cs.NE       14
cs.SI       12
eess.IV      8
cs.CY        4
cs.DC        4
cs.HC        4
cs.SD        4
cs.DB        3
Name: count, dtype: int64

Examples missing from API response:


,md_title,paper_url,arxiv_id


In [6]:
preview_cols = ["list_name", "topic", "md_title", "arxiv_title", "published", "primary_category", "arxiv_url"]
merged_df.sample(n=min(10, len(merged_df)), random_state=42)[preview_cols].reset_index(drop=True)

,list_name,topic,md_title,arxiv_title,published,primary_category,arxiv_url
0,The NLP,Information Retrieval and Text Mining,Relational World Knowledge Representation in C...,Relational World Knowledge Representation in C...,2021-04-12T21:50:55Z,cs.CL,http://arxiv.org/abs/2104.05837v2
1,The NLP,Question Answering,Tutorial on Answering Questions about Images w...,Tutorial on Answering Questions about Images w...,2016-10-04T16:29:28Z,cs.CV,http://arxiv.org/abs/1610.01076v1
2,The ML,"Classification, Clustering and Regression",A Survey of Classification Techniques in the A...,A Survey of Classification Techniques in the A...,2015-03-25T17:56:19Z,cs.LG,http://arxiv.org/abs/1503.07477v1
3,The ML,Architectures,Understanding LSTM - a tutorial into Long Shor...,Understanding LSTM -- a tutorial into Long Sho...,2019-09-12T15:44:51Z,cs.NE,http://arxiv.org/abs/1909.09586v1
4,The NLP,Generation,Keyphrase Generation: A Multi-Aspect Survey.,Keyphrase Generation: A Multi-Aspect Survey,2019-10-11T10:03:46Z,cs.CL,http://arxiv.org/abs/1910.05059v1
5,The ML,Generative Adversarial Networks,Deep Generative Modelling: A Comparative Revie...,Deep Generative Modelling: A Comparative Revie...,2021-03-08T17:34:03Z,cs.LG,http://arxiv.org/abs/2103.04922v4
6,The ML,Interpretability and Analysis,Explainable Artificial Intelligence Approaches...,Explainable Artificial Intelligence Approaches...,2021-01-23T06:15:34Z,cs.AI,http://arxiv.org/abs/2101.09429v1
7,The NLP,Large Language Models,"One Small Step for Generative AI, One Giant Le...","One Small Step for Generative AI, One Giant Le...",2023-04-04T06:22:09Z,cs.CY,http://arxiv.org/abs/2304.06488v1
8,The NLP,Information Retrieval and Text Mining,Taking Search to Task.,Taking Search to Task,2023-01-12T14:19:44Z,cs.IR,http://arxiv.org/abs/2301.05046v1
9,The NLP,Knowledge Graph,A Survey on Graph Neural Networks for Knowledg...,A Survey on Graph Neural Networks for Knowledg...,2020-07-24T06:46:46Z,cs.CL,http://arxiv.org/abs/2007.12374v1
